# Stage 18 — per-clip feature NORMALIZATION (spec item 2)
**One thing changed vs Stage 11:** positions are centred + scaled per clip (raw
MediaPipe [0,1] -> translation/scale invariant), vel/acc scaled consistently.
input_dim stays 190; same arch, same baseline augmentation. Targets the
per-signer spread 0.085->0.652.

**Baseline (Stage 11):** overall 0.4498 / lex 0.4348 / nonlex 0.5449; spread ~0.567.
Attach `wita-full-english-landmark-cache`. ~10 min reprocess + ~3 h train.


## Cell 1 — clone + deps


In [ ]:
%%capture
!pip install editdistance scipy --quiet
import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b stage13b-paper-replication "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')          # for `import wita_v2.*` (training)
sys.path.insert(0, '/kaggle/working/wita_v2')  # for `import stage18.*` (reprocess)
for _m in [m for m in list(sys.modules) if m.split('.')[0] in ('wita_v2','stage16','stage17','stage18')]:
    del sys.modules[_m]
import torch


In [ ]:
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


## Cell 2 — reprocess cache -> normalized (writable /kaggle/working)


In [ ]:
import os, glob
# EXPLICIT path -- no recursive glob.
SRC_CACHE  = '/kaggle/input/datasets/gaurs86/wita-full-english-landmark-cache/landmark_cache_122'
CACHE_ROOT = '/kaggle/working/landmark_cache_122_norm'
assert os.path.isdir(os.path.join(SRC_CACHE, 'train')), f'train/ not found under {SRC_CACHE} -- fix SRC_CACHE'
from stage18.normalize_cache import reprocess_cache
if not glob.glob(f'{CACHE_ROOT}/train/*/*.npz'):     # fixed-depth, NOT recursive
    reprocess_cache(SRC_CACHE, CACHE_ROOT)
print('normalized cache:', CACHE_ROOT, '| clips:', len(glob.glob(f'{CACHE_ROOT}/*/*/*.npz')))


## Cell 3 — config (identical to Stage 11; only LOG/CKPT dirs differ)


In [ ]:
import logging, random, numpy as np
from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig
VARIANT_NAME='stage18_norm'; LOG_DIR=f'/kaggle/working/logs_{VARIANT_NAME}'; CKPT_DIR=f'/kaggle/working/checkpoints_{VARIANT_NAME}'
os.makedirs(LOG_DIR,exist_ok=True); os.makedirs(CKPT_DIR,exist_ok=True)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-7s %(name)s — %(message)s',
                    handlers=[logging.StreamHandler(), logging.FileHandler(os.path.join(LOG_DIR,'log.log'))])
SEED=42; NUM_EPOCHS=80; BATCH_SIZE=32; LAMBDA_CTC=0.5; DEC_N_LAYERS=2; DEC_N_HEADS=4
LR_PEAK=5e-4; WEIGHT_DECAY=5e-2; GRAD_CLIP=1.0; DROPOUT=0.2; WARMUP_PCT=0.05
D_MODEL=256; N_LAYERS=4; N_HEADS=4; CONV_KERNEL=15; UPSAMPLE=2
cfg = Config(data=DataConfig(hf_repo_id='yewon816/WiTA', lang='english', max_zips=None, max_frames=64, seed=SEED),
             encoder=EncoderConfig(arch='siglip'),
             train=TrainConfig(num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr=LR_PEAK, weight_decay=WEIGHT_DECAY,
                               grad_clip=GRAD_CLIP, num_workers=2, warmup_pct=WARMUP_PCT, seed=SEED, checkpoint_dir=CKPT_DIR)).build()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.backends.cudnn.benchmark=True
print('variant', VARIANT_NAME, '| device', cfg.device)


## Cell 4 — train (normalized features; baseline augmentation, P_AFFINE=0)


In [ ]:
from wita_v2.training.stage11_train import train_stage11
from wita_v2.datasets.skeleton_augment import LandmarkAugment
train_aug = LandmarkAugment()   # Stage 1 v2 defaults (NO affine) -> isolates normalization
result = train_stage11(
    cache_root=CACHE_ROOT, cfg=cfg, num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    lr_peak=LR_PEAK, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP, dropout=DROPOUT,
    d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS, conv_kernel=CONV_KERNEL, upsample=UPSAMPLE,
    warmup_pct=WARMUP_PCT, dec_n_layers=DEC_N_LAYERS, dec_n_heads=DEC_N_HEADS,
    lambda_ctc=LAMBDA_CTC, label_smoothing=0.1, transform=train_aug, seed=SEED,
    checkpoint_dir=CKPT_DIR, log_dir=LOG_DIR, variant=VARIANT_NAME)
print('\n=== best on val ==='); print(json.dumps(result['best_payload'], indent=2, default=str))


## Cell 5 — test ONCE (marker-gated)


In [ ]:
from wita_v2.training.stage11_train import final_test_eval
test_out = final_test_eval(
    cache_root=CACHE_ROOT, checkpoint=result['checkpoint_path'], cfg=cfg, batch_size=BATCH_SIZE,
    d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS, conv_kernel=CONV_KERNEL, dropout=DROPOUT,
    upsample=UPSAMPLE, dec_n_layers=DEC_N_LAYERS, dec_n_heads=DEC_N_HEADS, log_dir=LOG_DIR, variant=VARIANT_NAME)


## Cell 6 — compare to Stage 11 + per-signer spread


In [ ]:
h = test_out['headline']
print(f"{'':12}{'overall':>9}{'lex':>9}{'nonlex':>9}")
print(f"{'Stage11':12}{0.4498:9.4f}{0.4348:9.4f}{0.5449:9.4f}")
print(f"{VARIANT_NAME:12}{h['test_overall_cer']:9.4f}{h['test_lex_cer']:9.4f}{h['test_nonlex_cer']:9.4f}")
print(f"{'paper':12}{0.2924:9.4f}{0.2810:9.4f}{0.3650:9.4f}")
ps=test_out['per_signer_cer']; v=sorted(ps.values())
print(f"\nper-signer: min={v[0]:.3f} max={v[-1]:.3f} spread={v[-1]-v[0]:.3f} (Stage11 ~0.567)")
print('Watch the spread + whether the hard signers improved (lower max).')
